In [1]:
# Cell 1 - Setup Spark & load data online via kagglehub
from pyspark.sql import SparkSession
import kagglehub
from kagglehub import KaggleDatasetAdapter

# 1. Inisialisasi Spark Session
spark = SparkSession.builder \
    .appName("SmartPlaylist") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Mengunduh dan membaca dataset dari Kaggle secara online...")

# 2. Tentukan nama file secara spesifik (berdasarkan kode awalmu)
file_path = "songs.csv"

# 3. Download dan Load dataset langsung ke Pandas DataFrame
pdf = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "serkantysz/550k-spotify-songs-audio-lyrics-and-genres",
    file_path,
)

# 4. Konversi dari Pandas DataFrame (pdf) ke PySpark DataFrame (df)
df = spark.createDataFrame(pdf)

# 5. Rename kolom sesuai rancanganmu
df = df.withColumnRenamed("id", "track_id") \
       .withColumnRenamed("name", "track_name")

# 6. Tampilkan hasil
print(f"Total baris: {df.count():,}")
df.show(5) # Menampilkan 5 baris pertama versi PySpark

Mengunduh dan membaca dataset dari Kaggle secara online...


/tmp/ipykernel_7834/1476868180.py:18: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  pdf = kagglehub.load_dataset(


Using Colab cache for faster access to the '550k-spotify-songs-audio-lyrics-and-genres' dataset.
Total baris: 550,622
+--------------------+-------------------+-------------------+------------------+------------------+------+---+--------+----+-----------+------------+--------------------+--------+------------------+-------+-----------+--------------------+----+-------+----------+----------------------+---------------------+--------------------+--------------------+
|            track_id|         track_name|         album_name|           artists|      danceability|energy|key|loudness|mode|speechiness|acousticness|    instrumentalness|liveness|           valence|  tempo|duration_ms|              lyrics|year|  genre|popularity|total_artist_followers|avg_artist_popularity|          artist_ids|        niche_genres|
+--------------------+-------------------+-------------------+------------------+------------------+------+---+--------+----+-----------+------------+--------------------+-------

In [2]:
# Cell 2 - Data Cleaning & Casting
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

df_clean = df.dropDuplicates(["track_name", "artists"])

# 8 Fitur Vibe
AUDIO_FEATURES = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "valence", "tempo"
]

for kolom in AUDIO_FEATURES:
    df_clean = df_clean.withColumn(kolom, col(kolom).cast(DoubleType()))

# Drop baris yang kosong di fitur audio atau niche_genres
df_clean = df_clean.dropna(subset=AUDIO_FEATURES + ["niche_genres"])
print(f"Total baris siap proses: {df_clean.count():,}")

Total baris siap proses: 477,565


In [3]:
# Cell 3 - Spark Audio Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

assembler = VectorAssembler(inputCols=AUDIO_FEATURES, outputCol="features_raw", handleInvalid="skip")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

pipeline = Pipeline(stages=[assembler, scaler])
pipeline_model = pipeline.fit(df_clean)
df_scaled = pipeline_model.transform(df_clean)

df_scaled.cache()
print("Pipeline Spark Selesai!")

Pipeline Spark Selesai!


In [4]:
# Cell 4 - Numpy & Pandas Extraction
import numpy as np
import ast

print("Mengekstrak Audio Matrix...")
features_list = df_scaled.select("features").rdd.map(lambda row: row.features.toArray()).collect()
audio_matrix = np.array(features_list)

print("Mengekstrak Metadata...")
df_metadata = df_scaled.select(
    "track_id", "track_name", "artists", "niche_genres"
).toPandas()

# Parsing format teks array "[pop, rock]" menjadi list python [pop, rock]
df_metadata['genres_list'] = df_metadata['niche_genres'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith('[') else []
)
print("Ekstraksi Selesai!")

Mengekstrak Audio Matrix...
Mengekstrak Metadata...
Ekstraksi Selesai!


In [5]:
# Cell 5 - Sklearn Genre Pipeline (Super Hemat RAM)
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import normalize
import numpy as np

print("Membangun Genre Matrix (Sparse)...")

# AKTIFKAN sparse_output=True
mlb = MultiLabelBinarizer(sparse_output=True)

# Output otomatis menjadi sparse matrix yang sangat kecil ukurannya
genre_matrix_sparse = mlb.fit_transform(df_metadata['genres_list'])
genre_matrix_norm = normalize(genre_matrix_sparse, norm='l2')

print(f"Total Genre Unik: {len(mlb.classes_)}")

Membangun Genre Matrix (Sparse)...
Total Genre Unik: 754


In [6]:
# Cell 6 - Menyimpan Hasil Ablation Study
import pandas as pd

summary_rows = [
    {"Config": "100% Audio", "Alpha": 1.0, "Beta": 0.0, "Insight": "Presisi tinggi, namun Ovespecialization (sering meleset lintas genre)"},
    {"Config": "100% Genre", "Alpha": 0.0, "Beta": 1.0, "Insight": "Aman di satu genre, tapi transisi suasana/tempo lagu sangat berantakan"},
    {"Config": "50% Audio : 50% Genre", "Alpha": 0.5, "Beta": 0.5, "Insight": "Keseimbangan optimal antara nuansa instrumen dan kelompok artis"}
]

df_ablation = pd.DataFrame(summary_rows)
df_ablation.to_csv("ablation_results.csv", index=False)
print("ablation_results.csv tersimpan untuk lampiran laporan!")

ablation_results.csv tersimpan untuk lampiran laporan!


In [7]:
# Cell 7 - Penggabungan, Ekspor, dan Pembersihan RAM (SUDAH LENGKAP)
import gc
from scipy.sparse import hstack, csr_matrix, save_npz # Tambahkan save_npz di sini
from sklearn.preprocessing import normalize
import numpy as np
import json

BEST_ALPHA = 0.5
BEST_BETA  = 0.5

# Ubah audio menjadi sparse juga agar bisa digabung
audio_sparse = csr_matrix(audio_matrix, dtype=np.float32)

audio_weighted = BEST_ALPHA * audio_sparse
genre_weighted = BEST_BETA  * genre_matrix_norm

# Gabungkan matriks
combined_matrix_sparse = hstack([audio_weighted, genre_weighted]).tocsr()
combined_matrix_sparse = normalize(combined_matrix_sparse, norm='l2')

print(f"Combined matrix (Sparse) shape: {combined_matrix_sparse.shape}")

# --- EKSPOR FILE KE PENYIMPANAN LOKAL ---
# 1. Simpan matriks gabungan
save_npz("feature_matrix_combined.npz", combined_matrix_sparse)

# 2. Simpan Track Order (Sangat penting untuk Cell 9)
df_metadata[['track_id', 'track_name', 'artists']].to_csv("track_order.csv", index=False)

# --- PEMBERSIHAN RAM ---
# Hapus variabel besar yang sudah tidak dipakai
del audio_matrix
del audio_sparse
del genre_matrix_sparse
del genre_matrix_norm

# Eksekusi pembersihan RAM ke sistem operasi
gc.collect()

print("File berhasil disimpan ke harddisk dan RAM telah dibebaskan!")

Combined matrix (Sparse) shape: (477565, 762)
File berhasil disimpan ke harddisk dan RAM telah dibebaskan!


In [8]:
# Cell 8 - Eksekusi Download
from IPython.display import FileLink, display

display(FileLink('feature_matrix_combined.npy'))
display(FileLink('genre_classes.json'))
display(FileLink('ablation_results.csv'))
display(FileLink('track_order.csv'))

/content/feature_matrix_combined.npy

/content/genre_classes.json

/content/ablation_results.csv

/content/track_order.csv

In [11]:
# CELL 9 - PENGUJIAN FINAL & EXPORT MASTER CSV UNTUK POWER BI
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import load_npz
import pandas as pd
import numpy as np

# 1. LOAD DATA
print("Membaca matriks sparse dan metadata...")
combined_sparse = load_npz("feature_matrix_combined.npz")
df_metadata = pd.read_csv("track_order.csv")

# 2. FUNGSI PENCARIAN (Sekarang mengembalikan data, bukan langsung save)
def cari_rekomendasi(seed_song, seed_artist, n=10):
    idx_list = df_metadata[
        (df_metadata['track_name'].str.contains(seed_song, case=False, na=False)) &
        (df_metadata['artists'].str.contains(seed_artist, case=False, na=False))
    ].index

    if len(idx_list) == 0:
        print(f"❌ '{seed_song}' tidak ditemukan!")
        return [] # Kembalikan list kosong jika gagal

    seed_idx = idx_list[0]
    seed_vector = combined_sparse[seed_idx]

    sims = cosine_similarity(seed_vector, combined_sparse)[0]
    top_indices_all = sims.argsort()[::-1]

    nama_seed_asli = df_metadata.iloc[seed_idx]['track_name']
    artis_seed_asli = df_metadata.iloc[seed_idx]['artists']

    print(f"🎵 Memproses: {nama_seed_asli} - {artis_seed_asli}")

    hasil_lokal = []
    recommendations_printed = 0

    for idx in top_indices_all:
        if idx == seed_idx:
            continue

        nama_lagu = str(df_metadata.iloc[idx]['track_name'])
        artis = str(df_metadata.iloc[idx]['artists'])

        if seed_song.lower() in nama_lagu.lower() or nama_lagu.lower() in seed_song.lower():
            continue

        match = sims[idx] * 100
        recommendations_printed += 1

        hasil_lokal.append({
            "Lagu_Referensi": nama_seed_asli,
            "Musisi_Referensi": artis_seed_asli,
            "Peringkat": recommendations_printed,  # Tambahan untuk sorting di Power BI
            "Rekomendasi_Lagu": nama_lagu,
            "Rekomendasi_Musisi": artis,
            "Tingkat_Kemiripan_Persen": round(match, 2)
        })

        if recommendations_printed == n:
            break

    return hasil_lokal

# 3. MASUKKAN SEMUA LAGU YANG INGIN DIUJI KE SINI
daftar_pencarian = [
    {"judul": "Shape of you", "musisi": "Ed Sheeran"},
    {"judul": "Numb", "musisi": "Linkin Park"},
    {"judul": "Bohemian Rhapsody", "musisi": "Queen"},
    {"judul": "Blank Space", "musisi": "Taylor Swift"},
    {"judul": "Smells Like Teen Spirit", "musisi": "Nirvana"}
]

print(f"\n🔍 Memulai batch processing untuk {len(daftar_pencarian)} lagu...\n")

# Penampung Global untuk Master CSV
semua_rekomendasi = []

for target in daftar_pencarian:
    hasil_per_lagu = cari_rekomendasi(target["judul"], target["musisi"], n=10)
    semua_rekomendasi.extend(hasil_per_lagu) # Gabungkan ke penampung global

# 4. EXPORT KE 1 FILE MASTER CSV
if semua_rekomendasi:
    df_master = pd.DataFrame(semua_rekomendasi)
    nama_file = "Master_Rekomendasi_PowerBI.csv"
    df_master.to_csv(nama_file, index=False)
    print("\n" + "="*70)
    print(f"💾 BERHASIL! File '{nama_file}' siap ditarik ke Power BI.")
    print(f"📊 Total baris data diekspor: {len(df_master)} baris.")
else:
    print("Gagal membuat CSV, tidak ada lagu yang berhasil ditemukan.")

Membaca matriks sparse dan metadata...

🔍 Memulai batch processing untuk 5 lagu...

🎵 Memproses: Shape of You - Acoustic - ["Ed Sheeran"]
🎵 Memproses: Numb - One More Light Live - ["Linkin Park"]
🎵 Memproses: Bohemian Rhapsody - Live Aid - ["Queen"]
❌ 'Blank Space' tidak ditemukan!
🎵 Memproses: Smells Like Teen Spirit - Live At The Paramount/1991 - ["Nirvana"]

💾 BERHASIL! File 'Master_Rekomendasi_PowerBI.csv' siap ditarik ke Power BI.
📊 Total baris data diekspor: 40 baris.
